In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [18]:
%%writefile mlp.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>
#include <time.h>

#define TILE 16

// Dimensions
const int N = 1024;
#define IN  512
#define H1  2048
#define H2  2048
#define H3  2048
#define OUT 512

// -------------------- CUDA ERROR CHECK --------------------
#define CHECK(call) \
{ \
    const cudaError_t error = call; \
    if (error != cudaSuccess) { \
        printf("Error: %s:%d, ", __FILE__, __LINE__); \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error)); \
        exit(1); \
    } \
}

// -------------------- INIT --------------------
void init_random(float *arr, int size) {
    for (int i = 0; i < size; i++)
        arr[i] = ((float)rand() / RAND_MAX) - 0.5f;
}

// -------------------- Forward Kernels --------------------
__global__ void matmul_bias(float* A,float* B,float* C,float* D,int M,int K,int N){
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    int numtiles=(K+TILE-1)/TILE;
    float acc=0.0f;
    for(int i=0;i<numtiles;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;

        if(Arow<M && Acol<K){
            sA[localrow][localcol]=A[Arow*K+Acol];
        }
        else{
            sA[localrow][localcol]=0.0f;
        }

        if(Brow<K && Bcol<N){
            sB[localrow][localcol]=B[Brow*N+Bcol];
        }
        else{
            sB[localrow][localcol]=0.0f;
        }
        __syncthreads();

        for(int k=0;k<TILE;k++){
            acc+=sA[localrow][k]*sB[k][localcol];
        }
        __syncthreads();
    }
    if(globalrow<M && globalcol<N){
        D[globalrow*N+globalcol]=acc+C[globalcol];
    }
}

__global__ void relu(float *A,float *B,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        B[tx]=fmaxf(0.0f,A[tx]);
    }
}

// -------------------- Backward Kernels --------------------

__global__ void matmulAtB(float *A,float *B,float *C,int M,int K,int N){
    //A(K,M) and B(K,N)
    int ArowT=threadIdx.y+blockIdx.y*blockDim.y;
    int Bcol=threadIdx.x+blockIdx.x*blockDim.x;

    float acc=0.0f;
    if(ArowT<M && Bcol<N){
        for(int i=0;i<K;i++){
            acc+=A[i*M+ArowT]*B[i*N+Bcol];
        }
        C[ArowT*N+Bcol]=acc;
    }
}

__global__ void matmulABt(float *A,float *B,float *C,int M,int K,int N){
    //A(M,K) and B(N,K)
    int Arow=threadIdx.y+blockIdx.y*blockDim.y;
    int BcolT=threadIdx.x+blockIdx.x*blockDim.x;

    float acc=0.0f;
    if(Arow<M && BcolT<N){
        for(int i=0;i<K;i++){
            acc+=A[Arow*K+i]*B[BcolT*K+i];
        }
        C[Arow*N+BcolT]=acc;
    }
}

__global__ void bias_backward(float*A,float*B,int M,int N){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<N){
        float acc=0.0f;
        for(int i=0;i<M;i++){
            acc+=A[i*N+tx];
        }
        B[tx]=acc;
    }
}

__global__ void relu_backward(float *A,float *B,float *C,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        C[tx] = (B[tx]>0) ? A[tx] : 0.0f; 
    }
}

__global__ void MSELoss(float *A,float *B,float *C,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        C[tx]=2*(A[tx]-B[tx])/M;
    }
}

// -------------------- SGD Optim --------------------

__global__ void sgd_update(float *param, float *grad, float lr, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < size) {
        param[idx] -= lr * grad[idx];
    }
}

int main(){
    srand(time(NULL));
    
    //Host allocations
    float* X=(float *)malloc(N*IN*sizeof(float));
    float *y = (float*)malloc(N*OUT*sizeof(float));

    float *W1 = (float*)malloc(IN*H1*sizeof(float));
    float *W2 = (float*)malloc(H1*H2*sizeof(float));
    float *W3 = (float*)malloc(H2*H3*sizeof(float));
    float *W4 = (float*)malloc(H3*OUT*sizeof(float));

    float *b1 = (float*)malloc(H1*sizeof(float));
    float *b2 = (float*)malloc(H2*sizeof(float));
    float *b3 = (float*)malloc(H3*sizeof(float));
    float *b4 = (float*)malloc(OUT*sizeof(float));

    //INIT
    init_random(X, N*IN);
    init_random(W1, IN*H1);
    init_random(W2, H1*H2);
    init_random(W3, H2*H3);
    init_random(W4, H3*OUT);
    init_random(b1, H1);
    init_random(b2, H2);
    init_random(b3, H3);
    init_random(b4, OUT);
    init_random(y, N*OUT);

    // Device allocations
    float *dX, *dZ1, *dA1, *dZ2, *dA2, *dZ3, *dA3, *dZ4;
    float *dW1, *dW2, *dW3, *dW4;
    float *db1, *db2, *db3, *db4;
    float *dY;

    //CUDA COPY
    CHECK(cudaMalloc(&dX, N*IN*sizeof(float)));
    CHECK(cudaMalloc(&dZ1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dA1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dZ2, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dA2,N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dZ3, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dA3, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dZ4, N*OUT*sizeof(float)));

    CHECK(cudaMalloc(&dW1, IN*H1*sizeof(float)));
    CHECK(cudaMalloc(&dW2, H1*H2*sizeof(float)));
    CHECK(cudaMalloc(&dW3, H2*H3*sizeof(float)));
    CHECK(cudaMalloc(&dW4, H3*OUT*sizeof(float)));

    CHECK(cudaMalloc(&db1, H1*sizeof(float)));
    CHECK(cudaMalloc(&db2, H2*sizeof(float)));
    CHECK(cudaMalloc(&db3, H3*sizeof(float)));
    CHECK(cudaMalloc(&db4, OUT*sizeof(float)));

    CHECK(cudaMalloc(&dY, N*OUT*sizeof(float)));

    CHECK(cudaMemcpy(dX, X, N*IN*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW1, W1, IN*H1*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW2, W2, H1*H2*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW3, W3, H2*H3*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW4, W4, H3*OUT*sizeof(float), cudaMemcpyHostToDevice));

    CHECK(cudaMemcpy(db1, b1, H1*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db2, b2, H2*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db3, b3, H3*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db4, b4, OUT*sizeof(float), cudaMemcpyHostToDevice));

    CHECK(cudaMemcpy(dY, y, N*OUT*sizeof(float), cudaMemcpyHostToDevice));

    dim3 threads(TILE,TILE);
    dim3 tl1((H1+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl2((H2+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl3((H3+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl4((OUT+TILE-1)/TILE,(N+TILE-1)/TILE);
    
    dim3 rel1(((N*H1)+256-1)/256);
    dim3 rel2(((N*H2)+256-1)/256);
    dim3 rel3(((N*H3)+256-1)/256);

    matmul_bias<<<tl1,threads>>>(dX,dW1,db1,dZ1,N,IN,H1);
    relu<<<rel1,256>>>(dZ1,dA1,N*H1);
    matmul_bias<<<tl2,threads>>>(dA1,dW2,db2,dZ2,N,H1,H2);
    relu<<<rel2,256>>>(dZ2,dA2,N*H2);
    matmul_bias<<<tl3,threads>>>(dA2,dW3,db3,dZ3,N,H2,H3);
    relu<<<rel3,256>>>(dZ3,dA3,N*H3);
    matmul_bias<<<tl4,threads>>>(dA3,dW4,db4,dZ4,N,H3,OUT);

    CHECK(cudaDeviceSynchronize());

    printf("Foward Pass Completed\n");

    //---------------------BackPass----------------------------

    float *dZ4g, *dA3g, *dZ3g, *dA2g, *dZ2g, *dA1g, *dZ1g;

    CHECK(cudaMalloc(&dZ4g, N*OUT*sizeof(float)));
    CHECK(cudaMalloc(&dA3g, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dZ3g, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dA2g, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dZ2g, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dA1g, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dZ1g, N*H1*sizeof(float)));

    float *dW1g, *dW2g, *dW3g, *dW4g;
    float *db1g, *db2g, *db3g, *db4g;
    
    CHECK(cudaMalloc(&dW1g, IN*H1*sizeof(float)));
    CHECK(cudaMalloc(&dW2g, H1*H2*sizeof(float)));
    CHECK(cudaMalloc(&dW3g, H2*H3*sizeof(float)));
    CHECK(cudaMalloc(&dW4g, H3*OUT*sizeof(float)));
    
    CHECK(cudaMalloc(&db1g, H1*sizeof(float)));
    CHECK(cudaMalloc(&db2g, H2*sizeof(float)));
    CHECK(cudaMalloc(&db3g, H3*sizeof(float)));
    CHECK(cudaMalloc(&db4g, OUT*sizeof(float)));

    dim3 tlb1((OUT+TILE-1)/TILE,(H3+TILE-1)/TILE);
    dim3 tlb2((H3+TILE-1)/TILE,(H2+TILE-1)/TILE);
    dim3 tlb3((H2+TILE-1)/TILE,(H1+TILE-1)/TILE);
    dim3 tlb4((H1+TILE-1)/TILE,(IN+TILE-1)/TILE);

    dim3 tbl1((H1+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tbl2((H2+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tbl3((H3+TILE-1)/TILE,(N+TILE-1)/TILE);

    dim3 tloss(((N*OUT)+256-1)/256);

    dim3 tbb1((OUT+256-1)/256);
    dim3 tbb2((H3+256-1)/256);
    dim3 tbb3((H2+256-1)/256);
    dim3 tbb4((H1+256-1)/256);

    dim3 tbrel1(((N*H3)+256-1)/256);
    dim3 tbrel2(((N*H2)+256-1)/256);
    dim3 tbrel3(((N*H1)+256-1)/256);

    MSELoss<<<tloss,256>>>(dZ4,dY,dZ4g,N*OUT);

    matmulAtB<<<tlb1,threads>>>(dA3,dZ4g,dW4g,H3,N,OUT);
    bias_backward<<<tbb1,256>>>(dZ4g,db4g,N,OUT);
    matmulABt<<<tbl3,threads>>>(dZ4g,dW4,dA3g,N,OUT,H3);

    CHECK(cudaDeviceSynchronize());

    relu_backward<<<tbrel1,256>>>(dA3g,dZ3,dZ3g,N*H3);

    CHECK(cudaDeviceSynchronize());

    matmulAtB<<<tlb2,threads>>>(dA2,dZ3g,dW3g,H2,N,H3);
    bias_backward<<<tbb2,256>>>(dZ3g,db3g,N,H3);
    matmulABt<<<tbl2,threads>>>(dZ3g,dW3,dA2g,N,H3,H2);

    CHECK(cudaDeviceSynchronize());

    relu_backward<<<tbrel2,256>>>(dA2g,dZ2,dZ2g,N*H2);

    CHECK(cudaDeviceSynchronize());

    matmulAtB<<<tlb3,threads>>>(dA1,dZ2g,dW2g,H1,N,H2);
    bias_backward<<<tbb3,256>>>(dZ2g,db2g,N,H2);
    matmulABt<<<tbl1,threads>>>(dZ2g,dW2,dA1g,N,H2,H1);

    CHECK(cudaDeviceSynchronize());

    relu_backward<<<tbrel3,256>>>(dA1g,dZ1,dZ1g,N*H1);

    CHECK(cudaDeviceSynchronize());

    matmulAtB<<<tlb4,threads>>>(dX,dZ1g,dW1g,IN,N,H1);
    bias_backward<<<tbb4,256>>>(dZ1g,db1g,N,H1);

    CHECK(cudaDeviceSynchronize());

    printf("Backward Pass Completed\n");

    //---------------------SGDupdate----------------------------

    float lr = 0.001f;
    int thread = 256;
    // W updates
    sgd_update<<<(IN*H1+255)/256, thread>>>(dW1, dW1g, lr, IN*H1);
    sgd_update<<<(H1*H2+255)/256, thread>>>(dW2, dW2g, lr, H1*H2);
    sgd_update<<<(H2*H3+255)/256, thread>>>(dW3, dW3g, lr, H2*H3);
    sgd_update<<<(H3*OUT+255)/256, thread>>>(dW4, dW4g, lr, H3*OUT);
    
    // b updates
    sgd_update<<<(H1+255)/256, thread>>>(db1, db1g, lr, H1);
    sgd_update<<<(H2+255)/256, thread>>>(db2, db2g, lr, H2);
    sgd_update<<<(H3+255)/256, thread>>>(db3, db3g, lr, H3);
    sgd_update<<<(OUT+255)/256, thread>>>(db4, db4g, lr, OUT);
    
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaGetLastError()); 

    printf("SGD update complete ✅\n");

    return 0;

    
}

Overwriting mlp.cu


In [19]:
!nvcc mlp.cu -o mlp

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!nvprof ./mlp

==2304== NVPROF is profiling process 2304, command: ./mlp
Foward Pass Completed
Backward Pass Completed
SGD update complete ✅
==2304== Profiling application: ./mlp
==2304== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   73.41%  282.66ms         3  94.220ms  42.472ms  161.89ms  matmulABt(float*, float*, float*, int, int, int)
                   12.34%  47.530ms         4  11.882ms  4.6731ms  19.023ms  matmul_bias(float*, float*, float*, float*, int, int, int)
                   11.58%  44.606ms         4  11.152ms  2.9378ms  24.252ms  matmulAtB(float*, float*, float*, int, int, int)
                    2.34%  9.0233ms        10  902.33us     832ns  3.4295ms  [CUDA memcpy HtoD]
                    0.13%  490.42us         8  61.302us  1.5030us  196.35us  sgd_update(float*, float*, float, int)
                    0.08%  301.98us         3  100.66us  93.726us  114.11us  relu_backward(float*, float*, float*, int)
       

In [7]:
%%writefile gradient_checkpoint.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>
#include <time.h>

#define TILE 16

// Dimensions
const int N = 1024;
#define IN  512
#define H1  2048
#define H2  2048
#define H3  2048
#define OUT 512

// -------------------- CUDA ERROR CHECK --------------------
#define CHECK(call) \
{ \
    const cudaError_t error = call; \
    if (error != cudaSuccess) { \
        printf("Error: %s:%d, ", __FILE__, __LINE__); \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error)); \
        exit(1); \
    } \
}

// -------------------- INIT --------------------
void init_random(float *arr, int size) {
    for (int i = 0; i < size; i++)
        arr[i] = ((float)rand() / RAND_MAX) - 0.5f;
}

// -------------------- Forward Kernels --------------------
__global__ void matmul_bias(float* A,float* B,float* C,float* D,int M,int K,int N){
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    int numtiles=(K+TILE-1)/TILE;
    float acc=0.0f;
    for(int i=0;i<numtiles;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;

        if(Arow<M && Acol<K){
            sA[localrow][localcol]=A[Arow*K+Acol];
        }
        else{
            sA[localrow][localcol]=0.0f;
        }

        if(Brow<K && Bcol<N){
            sB[localrow][localcol]=B[Brow*N+Bcol];
        }
        else{
            sB[localrow][localcol]=0.0f;
        }
        __syncthreads();

        for(int k=0;k<TILE;k++){
            acc+=sA[localrow][k]*sB[k][localcol];
        }
        __syncthreads();
    }
    if(globalrow<M && globalcol<N){
        D[globalrow*N+globalcol]=acc+C[globalcol];
    }
}

__global__ void relu(float *A,float *B,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        B[tx]=fmaxf(0.0f,A[tx]);
    }
}

// -------------------- Backward Kernels --------------------

__global__ void matmulAtB(float *A,float *B,float *C,int M,int K,int N){
    //A(K,M) and B(K,N)
    /*
    int ArowT=threadIdx.y+blockIdx.y*blockDim.y;
    int Bcol=threadIdx.x+blockIdx.x*blockDim.x;

    float acc=0.0f;
    if(ArowT<M && Bcol<N){
        for(int i=0;i<K;i++){
            acc+=A[i*M+ArowT]*B[i*N+Bcol];
        }
        C[ArowT*N+Bcol]=acc;
    }
    */
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    int numtiles=(K+TILE-1)/TILE;
    float acc=0.0f;
    for(int i=0;i<numtiles;i++){
        int row=i*TILE+localcol;

        if(row<K && globalrow<M){
            sA[localrow][localcol]=A[row*M + globalrow];
        }
        else{
            sA[localrow][localcol]=0.0f;
        }

        if(row<K && globalcol<N){
            sB[localrow][localcol]=B[row*N + globalcol];
        }
        else{
            sB[localrow][localcol]=0.0f;
        }
        __syncthreads();

        for(int k=0;k<TILE;k++){
            acc+=sA[localrow][k]*sB[k][localcol];
        }
        __syncthreads();
    }
    if(globalrow<M && globalcol<N){
        C[globalrow*N+globalcol]=acc;
    }
}

__global__ void matmulABt(float *A,float *B,float *C,int M,int K,int N){
    //A(M,K) and B(N,K)
    /*
    int Arow=threadIdx.y+blockIdx.y*blockDim.y;
    int BcolT=threadIdx.x+blockIdx.x*blockDim.x;

    float acc=0.0f;
    if(Arow<M && BcolT<N){
        for(int i=0;i<K;i++){
            acc+=A[Arow*K+i]*B[BcolT*K+i];
        }
        C[Arow*N+BcolT]=acc;
    }
    */
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;

    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    int numtiles=(K+TILE-1)/TILE;
    float acc=0.0f;
    for(int i=0;i<numtiles;i++){
        int col=i*TILE+localcol;

        if(col<K && globalrow<M){
            sA[localrow][localcol]=A[globalrow*K + col];
        }
        else{
            sA[localrow][localcol]=0.0f;
        }

        if(col<K && globalcol<N){
            sB[localrow][localcol]=B[globalcol*K + col];
        }
        else{
            sB[localrow][localcol]=0.0f;
        }
        __syncthreads();

        for(int k=0;k<TILE;k++){
            acc+=sA[localrow][k]*sB[k][localcol];
        }
        __syncthreads();
    }
    if(globalrow<M && globalcol<N){
        C[globalrow*N+globalcol]=acc;
    }
}

__global__ void bias_backward(float*A,float*B,int M,int N){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<N){
        float acc=0.0f;
        for(int i=0;i<M;i++){
            acc+=A[i*N+tx];
        }
        B[tx]=acc;
    }
}

__global__ void relu_backward(float *A,float *B,float *C,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        C[tx] = (B[tx]>0) ? A[tx] : 0.0f; 
    }
}

__global__ void MSELoss(float *A,float *B,float *C,int M){
    int tx=threadIdx.x+blockIdx.x*blockDim.x;
    if(tx<M){
        C[tx]=2*(A[tx]-B[tx])/M;
    }
}

// -------------------- SGD Optim --------------------

__global__ void sgd_update(float *param, float *grad, float lr, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < size) {
        param[idx] -= lr * grad[idx];
    }
}

int main(){
    srand(time(NULL));
    
    //Host allocations
    float* X=(float *)malloc(N*IN*sizeof(float));
    float *y = (float*)malloc(N*OUT*sizeof(float));

    float *W1 = (float*)malloc(IN*H1*sizeof(float));
    float *W2 = (float*)malloc(H1*H2*sizeof(float));
    float *W3 = (float*)malloc(H2*H3*sizeof(float));
    float *W4 = (float*)malloc(H3*OUT*sizeof(float));

    float *b1 = (float*)malloc(H1*sizeof(float));
    float *b2 = (float*)malloc(H2*sizeof(float));
    float *b3 = (float*)malloc(H3*sizeof(float));
    float *b4 = (float*)malloc(OUT*sizeof(float));

    //INIT
    init_random(X, N*IN);
    init_random(W1, IN*H1);
    init_random(W2, H1*H2);
    init_random(W3, H2*H3);
    init_random(W4, H3*OUT);
    init_random(b1, H1);
    init_random(b2, H2);
    init_random(b3, H3);
    init_random(b4, OUT);
    init_random(y, N*OUT);

    // Device allocations
    float *dX, *dZ1, *dA1, *dZ2, *dA2, *dZ3, *dA3, *dZ4;
    float *dW1, *dW2, *dW3, *dW4;
    float *db1, *db2, *db3, *db4;
    float *dY;

    //CUDA COPY
    CHECK(cudaMalloc(&dX, N*IN*sizeof(float)));
    CHECK(cudaMalloc(&dZ1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dA1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dZ2, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dA2,N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dZ3, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dA3, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dZ4, N*OUT*sizeof(float)));

    CHECK(cudaMalloc(&dW1, IN*H1*sizeof(float)));
    CHECK(cudaMalloc(&dW2, H1*H2*sizeof(float)));
    CHECK(cudaMalloc(&dW3, H2*H3*sizeof(float)));
    CHECK(cudaMalloc(&dW4, H3*OUT*sizeof(float)));

    CHECK(cudaMalloc(&db1, H1*sizeof(float)));
    CHECK(cudaMalloc(&db2, H2*sizeof(float)));
    CHECK(cudaMalloc(&db3, H3*sizeof(float)));
    CHECK(cudaMalloc(&db4, OUT*sizeof(float)));

    CHECK(cudaMalloc(&dY, N*OUT*sizeof(float)));

    CHECK(cudaMemcpy(dX, X, N*IN*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW1, W1, IN*H1*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW2, W2, H1*H2*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW3, W3, H2*H3*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dW4, W4, H3*OUT*sizeof(float), cudaMemcpyHostToDevice));

    CHECK(cudaMemcpy(db1, b1, H1*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db2, b2, H2*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db3, b3, H3*sizeof(float), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db4, b4, OUT*sizeof(float), cudaMemcpyHostToDevice));

    CHECK(cudaMemcpy(dY, y, N*OUT*sizeof(float), cudaMemcpyHostToDevice));

    dim3 threads(TILE,TILE);
    dim3 tl1((H1+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl2((H2+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl3((H3+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tl4((OUT+TILE-1)/TILE,(N+TILE-1)/TILE);
    
    dim3 rel1(((N*H1)+256-1)/256);
    dim3 rel2(((N*H2)+256-1)/256);
    dim3 rel3(((N*H3)+256-1)/256);

    matmul_bias<<<tl1,threads>>>(dX,dW1,db1,dZ1,N,IN,H1);
    relu<<<rel1,256>>>(dZ1,dA1,N*H1);
    matmul_bias<<<tl2,threads>>>(dA1,dW2,db2,dZ2,N,H1,H2);
    relu<<<rel2,256>>>(dZ2,dA2,N*H2);
    matmul_bias<<<tl3,threads>>>(dA2,dW3,db3,dZ3,N,H2,H3);
    relu<<<rel3,256>>>(dZ3,dA3,N*H3);
    matmul_bias<<<tl4,threads>>>(dA3,dW4,db4,dZ4,N,H3,OUT);

    CHECK(cudaDeviceSynchronize());

    printf("Foward Pass Completed\n");

    cudaFree(dA3);
    cudaFree(dZ3);
    cudaFree(dZ2);
    cudaFree(dA1);
    cudaFree(dZ1);

    //---------------------BackPass----------------------------

    float *dZ4g, *dA3g, *dZ3g, *dA2g, *dZ2g, *dA1g, *dZ1g;

    CHECK(cudaMalloc(&dZ4g, N*OUT*sizeof(float)));
    CHECK(cudaMalloc(&dA3g, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dZ3g, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dA2g, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dZ2g, N*H2*sizeof(float)));
    CHECK(cudaMalloc(&dA1g, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dZ1g, N*H1*sizeof(float)));

    float *dW1g, *dW2g, *dW3g, *dW4g;
    float *db1g, *db2g, *db3g, *db4g;
    
    CHECK(cudaMalloc(&dW1g, IN*H1*sizeof(float)));
    CHECK(cudaMalloc(&dW2g, H1*H2*sizeof(float)));
    CHECK(cudaMalloc(&dW3g, H2*H3*sizeof(float)));
    CHECK(cudaMalloc(&dW4g, H3*OUT*sizeof(float)));
    
    CHECK(cudaMalloc(&db1g, H1*sizeof(float)));
    CHECK(cudaMalloc(&db2g, H2*sizeof(float)));
    CHECK(cudaMalloc(&db3g, H3*sizeof(float)));
    CHECK(cudaMalloc(&db4g, OUT*sizeof(float)));

    dim3 tlb1((OUT+TILE-1)/TILE,(H3+TILE-1)/TILE);
    dim3 tlb2((H3+TILE-1)/TILE,(H2+TILE-1)/TILE);
    dim3 tlb3((H2+TILE-1)/TILE,(H1+TILE-1)/TILE);
    dim3 tlb4((H1+TILE-1)/TILE,(IN+TILE-1)/TILE);

    dim3 tbl1((H1+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tbl2((H2+TILE-1)/TILE,(N+TILE-1)/TILE);
    dim3 tbl3((H3+TILE-1)/TILE,(N+TILE-1)/TILE);

    dim3 tloss(((N*OUT)+256-1)/256);

    dim3 tbb1((OUT+256-1)/256);
    dim3 tbb2((H3+256-1)/256);
    dim3 tbb3((H2+256-1)/256);
    dim3 tbb4((H1+256-1)/256);

    dim3 tbrel1(((N*H3)+256-1)/256);
    dim3 tbrel2(((N*H2)+256-1)/256);
    dim3 tbrel3(((N*H1)+256-1)/256);

    MSELoss<<<tloss,256>>>(dZ4,dY,dZ4g,N*OUT);
    cudaFree(dZ4);

    CHECK(cudaMalloc(&dZ3, N*H3*sizeof(float)));
    CHECK(cudaMalloc(&dA3, N*H3*sizeof(float)));

    matmul_bias<<<tl3,threads>>>(dA2,dW3,db3,dZ3,N,H2,H3);
    relu<<<rel3,256>>>(dZ3,dA3,N*H3);

    matmulAtB<<<tlb1,threads>>>(dA3,dZ4g,dW4g,H3,N,OUT);
    bias_backward<<<tbb1,256>>>(dZ4g,db4g,N,OUT);
    matmulABt<<<tbl3,threads>>>(dZ4g,dW4,dA3g,N,OUT,H3);
    cudaFree(dA3);
    relu_backward<<<tbrel1,256>>>(dA3g,dZ3,dZ3g,N*H3);
    cudaFree(dZ3);

    matmulAtB<<<tlb2,threads>>>(dA2,dZ3g,dW3g,H2,N,H3);
    bias_backward<<<tbb2,256>>>(dZ3g,db3g,N,H3);
    matmulABt<<<tbl2,threads>>>(dZ3g,dW3,dA2g,N,H3,H2);

    CHECK(cudaMalloc(&dZ1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dA1, N*H1*sizeof(float)));
    CHECK(cudaMalloc(&dZ2, N*H2*sizeof(float)));
    matmul_bias<<<tl1,threads>>>(dX,dW1,db1,dZ1,N,IN,H1);
    relu<<<rel1,256>>>(dZ1,dA1,N*H1);
    matmul_bias<<<tl2,threads>>>(dA1,dW2,db2,dZ2,N,H1,H2);

    relu_backward<<<tbrel2,256>>>(dA2g,dZ2,dZ2g,N*H2);
    cudaFree(dZ2);

    matmulAtB<<<tlb3,threads>>>(dA1,dZ2g,dW2g,H1,N,H2);
    bias_backward<<<tbb3,256>>>(dZ2g,db2g,N,H2);
    matmulABt<<<tbl1,threads>>>(dZ2g,dW2,dA1g,N,H2,H1);
    cudaFree(dA1);

    relu_backward<<<tbrel3,256>>>(dA1g,dZ1,dZ1g,N*H1);
    cudaFree(dZ1);

    matmulAtB<<<tlb4,threads>>>(dX,dZ1g,dW1g,IN,N,H1);
    bias_backward<<<tbb4,256>>>(dZ1g,db1g,N,H1);

    CHECK(cudaDeviceSynchronize());

    printf("Backward Pass Completed\n");

    //---------------------SGDupdate----------------------------

    float lr = 0.001f;
    int thread = 256;
    // W updates
    sgd_update<<<(IN*H1+255)/256, thread>>>(dW1, dW1g, lr, IN*H1);
    sgd_update<<<(H1*H2+255)/256, thread>>>(dW2, dW2g, lr, H1*H2);
    sgd_update<<<(H2*H3+255)/256, thread>>>(dW3, dW3g, lr, H2*H3);
    sgd_update<<<(H3*OUT+255)/256, thread>>>(dW4, dW4g, lr, H3*OUT);
    
    // b updates
    sgd_update<<<(H1+255)/256, thread>>>(db1, db1g, lr, H1);
    sgd_update<<<(H2+255)/256, thread>>>(db2, db2g, lr, H2);
    sgd_update<<<(H3+255)/256, thread>>>(db3, db3g, lr, H3);
    sgd_update<<<(OUT+255)/256, thread>>>(db4, db4g, lr, OUT);
    
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaGetLastError()); 

    printf("SGD update complete ✅\n");

    cudaFree(dX);
    cudaFree(dY);

    return 0;

    
}

Overwriting gradient_checkpoint.cu


In [8]:
!nvcc gradient_checkpoint.cu -o gc

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [9]:
!nvprof ./gc

==339== NVPROF is profiling process 339, command: ./gc
Foward Pass Completed
Backward Pass Completed
SGD update complete ✅
==339== Profiling application: ./gc
==339== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   42.38%  93.273ms         7  13.325ms  4.8726ms  19.847ms  matmul_bias(float*, float*, float*, float*, int, int, int)
                   32.22%  70.911ms         4  17.728ms  6.9041ms  29.236ms  matmulAtB(float*, float*, float*, int, int, int)
                   20.51%  45.149ms         3  15.050ms  5.1450ms  20.639ms  matmulABt(float*, float*, float*, int, int, int)
                    4.22%  9.2817ms        10  928.17us     800ns  3.6018ms  [CUDA memcpy HtoD]
                    0.22%  493.24us         8  61.655us  2.7200us  193.44us  sgd_update(float*, float*, float, int)
                    0.17%  380.35us         5  76.069us  75.584us  76.607us  relu(float*, float*, int)
                    0.16%  352